**Purpose:** Bridge stage: turns raw 01_data files into the experiment-ready `*_0` inputs (etfs_0.pkl, news_0.json, reddit_0.parquet).

**Inputs:** `02_sentiment/reddit/dataset.parquet`

**Outputs:** `reddit_data.parquet`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT


In [ ]:
def load_etfs_data(filename = "etfs_0.pkl"):
    import pickle

    # # to save into the same format
    # with open("etfs_data.pkl", "wb") as f:
    #     pickle.dump(etfs, f)

    with open(filename, "rb") as f:
        etfs = pickle.load(f)
    
    return etfs

In [ ]:
def load_reddit_data(filename = "reddit_0.parquet"):
    import pandas as pd
    import json
    
    # # to save into the same format
    # df["top_comments"] = df["top_comments"].apply(json.dumps)
    # df["submission"] = df["submission"].apply(json.dumps)
    # df.to_parquet("reddit_data.parquet", index=True)

    df = pd.read_parquet(filename)
    df["top_comments"] = df["top_comments"].apply(json.loads)
    df["submission"] = df["submission"].apply(json.loads)

    return df

In [ ]:
def load_news_data(timestamp, settings="news_0.json"):
    import json
    import requests, zipfile, io
    import re
    import pandas as pd
    import spacy
    nlp = spacy.load("en_core_web_sm")

    def clean_quotations(text):
        text = re.sub(r'\d+\|\d+\|\|', '', text)
        text = re.sub(r'#\d+', '', text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\s([.,!?;:])', r'\1', text)
        
        return text.strip()
    
    def lemmatize_quotation(text):
        doc = nlp(text)
        lemmas = [token.lemma_ for token in doc]

        return " ".join(lemmas)
    
    with open(settings, "r") as f:
        news_settings = json.load(f)

    if timestamp not in news_settings["DATE_filter"]:
        raise ValueError("Timestamp not in settings DATE_filter")

    sector_patterns = {
        sector: r"(?<!\w)(?:" + "|".join(map(re.escape, words)) + r")(?!\w)"
        for sector, words in news_settings["Quotations_filter"].items()
    }

    sources = {source for lst in news_settings["SourceCommonName_filter"].values() for source in lst}


    #####################
    # url specific part #
    
    url = f"http://data.gdeltproject.org/gdeltv2/{timestamp}.gkg.csv.zip"
    r = requests.get(url)

    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            with z.open(z.namelist()[0]) as f:
                df = pd.read_csv(f, sep='\t', header=None, dtype=str, encoding='ISO-8859-1')
                df.columns = [
                    "GKGRECORDID", "DATE", "SourceCollectionIdentifier", "SourceCommonName",
                    "DocumentIdentifier", "Counts", "V2Counts", "Themes", "V2Themes",
                    "Locations", "V2Locations", "Persons", "V2Persons", "Organizations",
                    "V2Organizations", "V2Tone", "Dates", "GCAM", "SharingImage",
                    "RelatedImages", "SocialImageEmbeds", "SocialVideoEmbeds", "Quotations",
                    "AllNames", "Amounts", "TranslationInfo", "Extras"
                ]
                df = df[["GKGRECORDID", "DATE", "SourceCommonName", "Quotations"]]
                df = df[df["SourceCommonName"].isin(sources)]
                # df = df[df["Quotations"].apply(lambda x: isinstance(x, str))] # replaced with:
                df = df[df["Quotations"].notna()]
                df["Quotations"] = df["Quotations"].map(clean_quotations)
                df = df.drop_duplicates(subset=["SourceCommonName", "Quotations"])
                df["search_Quotations"] = df["Quotations"].map(lemmatize_quotation)

                sector_dfs = {}
                for sector, pattern in sector_patterns.items():

                    sector_dfs[sector] = df[df["SourceCommonName"].isin(news_settings["SourceCommonName_filter"][sector])]
                    sector_dfs[sector] = sector_dfs[sector][sector_dfs[sector]["search_Quotations"].str.contains(pattern, na=False, flags=re.IGNORECASE)]

    except Exception as e:
        error_message = f"Failed to process {url}: {e}\n"
        with open("errors.log", "a") as log_file:
            log_file.write(error_message)
        return False

    return sector_dfs

In [ ]:
# load_etfs_data()
# load_reddit_data()
# load_news_data("20240629151500")

# news labelling validation 

In [ ]:
import os
dfs = [x for x in os.listdir(str(PROJECT_ROOT / "02_sentiment/labeling")) if x.endswith(".parquet") and x.startswith("news_")]


dfs_buckets = {}
for df_name in dfs:
    name = df_name.split("_")
    news, model, bucket, last_hash = name[0], name[1], name[2], int(name[3].replace(".parquet", ""))
    
    # get bucket
    if bucket not in dfs_buckets:
        dfs_buckets[bucket] = {}
    
    if model not in dfs_buckets[bucket]:
        dfs_buckets[bucket][model] = []

    dfs_buckets[bucket][model].append(last_hash)

for bucket in dfs_buckets:
    for model in dfs_buckets[bucket]:
        dfs_buckets[bucket][model] = max(dfs_buckets[bucket][model]) #/ 14719120652134250070558007437148545

for bucket in dfs_buckets:

    # get min last_hash across models for the bucket
    min_last_hash = min(dfs_buckets[bucket].values())

    # get corresponding model
    for model in dfs_buckets[bucket]:
        if dfs_buckets[bucket][model] == min_last_hash:
            print(f"Bucket: {bucket}, Model: {model}, Last Hash: {min_last_hash}")

dfs_buckets

In [ ]:
import pandas as pd
import numpy as np
import os

dfs = [x for x in os.listdir(str(PROJECT_ROOT / "02_sentiment/labeling")) if x.endswith(".parquet") and x.startswith("news_")]

expected_len = sum([len(df) for df in [pd.read_parquet(os.path.join("label", df_name)) for df_name in dfs]])
concated_df = pd.concat([x for x in [pd.read_parquet(os.path.join("label", df_name)) for df_name in dfs]], ignore_index=True)
print(f"{expected_len==len(concated_df)} : Validate expected length of concatenated DataFrame")

GKGRECORDID_above_3 = sum(pd.Series(concated_df["GKGRECORDID"].value_counts().values) > 3)
print(f"{GKGRECORDID_above_3 == 0} : Validate no ({GKGRECORDID_above_3}) GKGRECORDID appears more than 3 times")

print()

df_grouped = (
    concated_df.groupby("GKGRECORDID", as_index=False)
      .agg(lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan)
)
original_len = len(df_grouped)
df_grouped.dropna(inplace=True)
dropped_len = original_len - len(df_grouped)
print(f"Dropped {dropped_len} rows with NaN values after grouping.")
df_grouped

In [ ]:
sectors = {"Energy", "Materials", "Industrials", "Consumer Discretionary", "Consumer Staples", "Health Care", "Financials", "Information Technology", "Communication Services", "Utilities", "Real Estate"}
ranks = {sector: {"pos3": 0, "pos2": 0, "pos1": 0, " ": " ", "neu3": 0, "neu2": 0, "neu1": 0, "": "", "neg3": 0, "neg2": 0, "neg1": 0} for sector in sectors}


for i, row in df_grouped.iterrows():
    a = row["google/gemma-7b-it"]
    b = row["Qwen/Qwen2.5-7B-Instruct"]
    c = row["meta-llama/Llama-3.1-8B-Instruct"]

    pos_count = {}
    neg_count = {}

    sector = set([x for x in [a.get("sector", None), b.get("sector", None), c.get("sector", None)] if x is not None])
    if len(sector) != 1:
        #print("Sector mismatch or missing:", sector, " at ", row["GKGRECORDID"])
        continue
    sector = sector.pop()

    votes = [a.get("sentiment", None), b.get("sentiment", None), c.get("sentiment", None)]
    pos_votes = votes.count("positive")
    neg_votes = votes.count("negative")
    neu_votes = votes.count("neutral")

    if pos_votes == 3:
        ranks[sector]["pos3"] += 1
    elif pos_votes == 2:
        ranks[sector]["pos2"] += 1
    elif pos_votes == 1:
        ranks[sector]["pos1"] += 1

    if neu_votes == 3:
        ranks[sector]["neu3"] += 1
    elif neu_votes == 2:
        ranks[sector]["neu2"] += 1
    elif neu_votes == 1:
        ranks[sector]["neu1"] += 1

    if neg_votes == 3:
        ranks[sector]["neg3"] += 1
    elif neg_votes == 2:
        ranks[sector]["neg2"] += 1
    elif neg_votes == 1:
        ranks[sector]["neg1"] += 1
    
ranks_df = pd.DataFrame.from_dict(ranks, orient="index")
ranks_df

In [ ]:
ranks_df.sum(axis=0)

In [ ]:
ranks_df["pos"] = ranks_df["pos3"] + ranks_df["pos2"]
ranks_df["neu"] = ranks_df["neu3"] + ranks_df["neu2"]
ranks_df["neg"] = ranks_df["neg3"] + ranks_df["neg2"]

ranks_df[["pos", "neu", "neg"]]

In [ ]:
# import random
# import json
# from collections import defaultdict

# TARGET_PER_GROUP = 20
# SENTIMENTS = ["positive", "neutral", "negative"]

# # output structure
# output = defaultdict(
#     lambda: {
#         s: {"2": [], "3": []} for s in SENTIMENTS
#     }
# )

# # lazy cache: {(date, sector): dataframe}
# news_cache = {}

# def get_quotation(gkrecordid, sector):
#     date = gkrecordid.split("-")[0]
#     key = (date, sector)

#     if key not in news_cache:
#         try:
#             news_cache[key] = load_news_data(date)[sector]
#         except KeyError:
#             news_cache[key] = None

#     df = news_cache[key]
#     if df is None:
#         return None

#     match = df[df["GKGRECORDID"] == gkrecordid]
#     if match.empty:
#         return None

#     return match["Quotations"].values[0]


# def vote_stats(row):
#     votes = [
#         row["google/gemma-7b-it"].get("sentiment"),
#         row["Qwen/Qwen2.5-7B-Instruct"].get("sentiment"),
#         row["meta-llama/Llama-3.1-8B-Instruct"].get("sentiment"),
#     ]
#     return {
#         "positive": votes.count("positive"),
#         "neutral": votes.count("neutral"),
#         "negative": votes.count("negative"),
#     }


# # shuffle once for global randomness
# rows = list(df_grouped.iterrows())
# random.shuffle(rows)

# for _, row in rows:
#     gkrecordid, sector = row["GKGRECORDID"].split(maxsplit=1)
#     sector = sector.replace("(", "").replace(")", "")

#     stats = vote_stats(row)

#     for sentiment in SENTIMENTS:
#         count = stats[sentiment]

#         if count not in (2, 3):
#             continue

#         bucket = output[sector][sentiment][str(count)]
#         if len(bucket) >= TARGET_PER_GROUP:
#             continue

#         quotation = get_quotation(gkrecordid, sector)
#         if quotation is None:
#             continue

#         bucket.append({
#             "gkrecordid": gkrecordid,
#             "quotation": quotation,
#             "label": "todo"
#         })


# # remove empty sectors (optional)
# output = {
#     sector: data
#     for sector, data in output.items()
#     if any(
#         len(data[s][k]) > 0
#         for s in SENTIMENTS
#         for k in ("2", "3")
#     )
# }

# # save
# with open("sector_sentiment_quotations.json", "w", encoding="utf-8") as f:
#     json.dump(output, f, indent=2, ensure_ascii=False)

# print("✅ JSON created with lazy-loaded news data")


In [ ]:
# import json

# with open("sector_sentiment_quotations.json", "r", encoding="utf-8") as f:
#     sector_quotations = json.load(f)

# total = 0
# for sector, items in sector_quotations.items():
#     for sentiment, item in sector_quotations[sector].items():
#         for majority, item in sector_quotations[sector][sentiment].items():
#             for quote in item:
#                 total += 1
# total == (11 * 3 * 2 * 20) # 11 sectors, 3 sentiments, 2 majority levels (3v0 or 2v1), 20 quotes each

In [ ]:
# import requests
# import json
# import re
# import time

# models_used = set()
# done_counter = 0

# with open(str(PROJECT_ROOT / "01_data/processed/sector_gpt_api_labeled.json"), "r") as f:
#     sector_gpt_api = json.load(f)

# import uuid



# for sector in sector_gpt_api:
#     for sentiment_loop in sector_gpt_api[sector]:
#         for majority in sector_gpt_api[sector][sentiment_loop]:
#             for i, item in enumerate(sector_gpt_api[sector][sentiment_loop][majority]):

#                 if item["label"] != "todo":
#                     continue

#                 endpoint = "https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream"

#                 headers = {
#                     "x-api-key": IAEDU_API_KEY  # from src.secrets (see src/secrets_example.py)
#                 }

#                 def prompt_template(news_sector, news_quote):
#                     x = (
#                         "You are a financial-domain impact classifier for equity sectors.\n"
#                         "Your task is to analyze a GDELT news quote and classify the expected impact of the described events on a specific GICS sector.\n"
#                         "\n"
#                         "Classification Rules:\n"
#                         '- "positive": the event is expected to benefit the sector (e.g., increased demand, revenue growth, cost reduction, regulatory advantage, market share gain).\n'
#                         '- "negative": the event is expected to harm the sector (e.g., decreased demand, revenue loss, higher costs, regulatory pressure, operational disruption).\n'
#                         '- "neutral": the event has no meaningful impact, is unrelated, or the impact is ambiguous/uncertain.\n'
#                         "\n"
#                         'Definition of "meaningful impact":\n'
#                         "- There must be an explicit or implied causal link between the event and sector performance.\n"
#                         "- The event should clearly suggest an effect on revenue, costs, operations, regulatory conditions, or valuation.\n"
#                         "- Interpretation is allowed, but it must be logically justified with cause-effect reasoning.\n"
#                         "\n"
#                         "Non-meaningful impact:\n"
#                         "- Mere mentions of companies, sectors, or locations without linkage to sector performance.\n"
#                         "- Facts, statistics, or descriptions without explanation of consequences.\n"
#                         "- Speculative statements with no plausible sector-level effect.\n"
#                         "Output Requirements:\n"
#                         "Return a single JSON object **exactly** in the following format:\n"
#                         "{\n"
#                         '   "sector": "<sector name>",\n'
#                         '   "sentiment": "<positive | negative | neutral>",\n'
#                         '   "rationale": "<3–5 sentence explanation linking the event to sector impact>",\n'
#                         '   "confidence": <float between 0 and 1>\n'
#                         "}"
#                         "\n"
#                         '- "rationale": 2–5 sentence explanation of why the impact label was chosen, highlighting the causal link.\n'
#                         '- "confidence": float between 0.0 (very unsure) to 1.0 (very confident) representing your certainty.\n'
#                         '- Default label is "neutral" if impact is unclear or not meaningful.\n'
#                         "\n"
#                         "Content to classify:\n"
#                         f"Sector: {news_sector}\n"
#                         f"Quote: {news_quote}\n"
#                         "\n"
#                         "--- END OF CONTENT ---\n"
#                         "Classify strictly according to these rules. Output only the JSON object, nothing else."
#                     )
#                     return x

#                 thread_id = str(uuid.uuid4())
#                 data = {
#                     "channel_id": "cmkpzn9wrjui7gr01gz7qvdsu",
#                     "thread_id": thread_id,
#                     "user_info": "{}",
#                     "message": prompt_template(
#                         news_sector=sector,
#                         news_quote=item["quotation"]
#                     )
#                 }

#                 response = requests.post(
#                     endpoint,
#                     headers=headers,
#                     data=data,
#                     stream=True
#                 )


#                 final_answer = None

#                 for line in response.iter_lines(decode_unicode=True):
#                     if not line:
#                         continue

#                     event = json.loads(line)

#                     if event["type"] == "message":
#                         final_answer = event["content"]["content"]
#                         model_name = event["content"]["response_metadata"]["model_name"]

#                     elif event["type"] == "done":
#                         break

#                     # print(event)
#                 try:
#                     anserw = final_answer
#                     match = re.search(
#                         r'"sentiment"\s*:\s*"(positive|negative|neutral)"',
#                         anserw,
#                         re.IGNORECASE
#                     )

#                     sentiment = match.group(1).lower() if match else "todo"
                
                    
#                 except Exception as e:
#                     sentiment = "todo"

#                 if sentiment == "todo":
#                     print(f"Could not extract sentiment for item: {item} at {time.strftime('%H:%M:%S', time.gmtime())}.")
#                     continue

#                 else:
#                     sector_gpt_api[sector][sentiment_loop][majority][i]["label"] = sentiment

#                 done_counter += 1
#                 print(f"Done items: {done_counter} ({sentiment})", end="\r")
#                 models_used.add(model_name)



# with open(str(PROJECT_ROOT / "01_data/processed/sector_gpt_api_labeled_FINAL.json"), "w") as f:
#     json.dump(sector_gpt_api, f, indent=2)
    
# print("Models used:", models_used)

print("Models used: {'gpt-4o-2024-11-20'}")

In [ ]:
# import json
# import pandas as pd

# with open("sector_sentiment_quotations.json", "r", encoding="utf-8") as f:
#     sector_quotations = json.load(f)

# rows = []

# for sector, items in sector_quotations.items():
#     for sentiment, item in sector_quotations[sector].items():
#         for majority, item in sector_quotations[sector][sentiment].items():
#             for quote in item:

#                 label = quote["label"]
                
#                 rows.append({
#                     "gkrecordid": quote["gkrecordid"],
#                     "quote": quote["quotation"],
#                     "sector": sector,
#                     "llms_sentiment": sentiment,
#                     "llms_agreement": majority,
#                     "gpt_sentiment": label,
#                 })

# df_labels = pd.DataFrame(rows)


# df = df_labels.copy()

# # 2. Match Logic
# df['is_match'] = df['gpt_sentiment'] == df['llms_sentiment']

# # for i, row in df.iterrows():
# #     if row['is_match']:
# #         continue

# #     if row["llms_agreement"] != "3":
# #         continue

# #     print("\n" + "=" * 80)
# #     print(f"GKGRECORDID: {row['gkrecordid']}\n")
# #     print(f"Sector: {row['sector']}")
# #     print(f"Expected LLMs Sentiment: {row['llms_sentiment']} (Agreement: {row['llms_agreement']}) but got GPT Sentiment: {row['gpt_sentiment']}")

# #     print("\nLabel seems to disagree. Please re-evaluate the quotation:")
# #     quotation = row['quote']
# #     print(quotation)

01_data/processed/sector_gpt_api_labeled_FINAL.json was using {'gpt-4o-2024-11-20'} api

01_data/processed/sector_sentiment_quotations.json was using chatgpt 5.2 flagship model (?) via copy pasting in the browser

In [ ]:
import json
import pandas as pd

with open("sector_sentiment_quotations.json", "r", encoding="utf-8") as f:
    sector_quotations = json.load(f)

# with open(str(PROJECT_ROOT / "01_data/processed/sector_gpt_api_labeled_FINAL.json"), "r", encoding="utf-8") as f:
#     sector_quotations = json.load(f)

rows = []

for sector, items in sector_quotations.items():
    for sentiment, item in sector_quotations[sector].items():
        for majority, item in sector_quotations[sector][sentiment].items():
            for quote in item:

                label = quote["label"]
                
                rows.append({
                    "sector": sector,
                    "llms_sentiment": sentiment,
                    "llms_agreement": majority,
                    "gpt_sentiment": label,
                })

df_labels = pd.DataFrame(rows)


df = df_labels.copy()

# 2. Match Logic
df['is_match'] = df['gpt_sentiment'] == df['llms_sentiment']

# 3. Aggregate results
summary = df.groupby(['sector', 'llms_sentiment', 'llms_agreement']).agg(
    matches=('is_match', 'sum'),
    total=('is_match', 'count')
).reset_index()

summary['pct'] = (summary['matches'] / summary['total'] * 100).round(1).astype(str) + '% | ' + summary['matches'].astype(str) + '/' + summary['total'].astype(str)

# 4. Create Pivot Table
# This spreads the 'llms_agreement' across columns for easy comparison
pivot_table = summary.pivot_table(
    index=['sector', 'llms_sentiment'],
    columns='llms_agreement',
    values=['pct'],
    aggfunc='first'
).fillna('-')




mean_pct = df_labels.groupby(['llms_agreement']).apply(lambda x: (x['gpt_sentiment'] == x['llms_sentiment']).mean() * 100).round(1)
sum_numerators = df_labels.groupby(['llms_agreement']).apply(lambda x: (x['gpt_sentiment'] == x['llms_sentiment']).sum())
sum_denominators = df_labels.groupby(['llms_agreement']).size()

print("Mean % per column:")
print(mean_pct)
print("\nSum of numerators per column:")
print(sum_numerators)
print("\nSum of denominators per column:")
print(sum_denominators)

pivot_table

In [ ]:
import os
import pandas as pd

PARQUET_PATH = "cache/news00001.parquet"

if os.path.exists(PARQUET_PATH):
    existing_df = pd.read_parquet(PARQUET_PATH)
    existing_ids = set(existing_df["id"])
else:
    existing_df = None
    existing_ids = set()


quotes_df = {
    "id": [],
    "quotation": [],
}

df_grouped = df_grouped.sort_values("GKGRECORDID")
current_timestamp = ("0", None)

import random

for _, row in df_grouped.iterrows():

    id, sector = row["GKGRECORDID"].split(maxsplit=1)

    if id in existing_ids:
        continue

    date = id.split("-")[0]
    sector = sector.replace("(", "").replace(")", "")

    if date != current_timestamp[0]:
        current_timestamp = (date, load_news_data(date))

    data = current_timestamp[1][sector]
    quotation = data.loc[
        data["GKGRECORDID"] == id, "Quotations"
    ].values[0]

    quotes_df["id"].append(id)
    quotes_df["quotation"].append(quotation)
    existing_ids.add(id)

if existing_df is not None:
    quotes_df = pd.DataFrame(quotes_df)
    quotes_df = pd.concat([existing_df, quotes_df], ignore_index=True)

# pd.DataFrame(quotes_df).drop_duplicates("id").to_parquet(
#     PARQUET_PATH, index=False
# )

# print("14792", "16360", "18529", "20974")
# pd.read_parquet(PARQUET_PATH)

# reddit labelling validation

In [ ]:
import os
import pandas as pd

files = [x for x in os.listdir(str(PROJECT_ROOT / "02_sentiment/labeling")) if x.startswith("reddit_") and x.endswith(".parquet")]

dfs = {"Qwen": [], "meta-llama": [], "google": []}
for file in files:
    key = file.split("_")[1]
    df = pd.read_parquet(os.path.join("label", file))
    dfs[key].append(df)

for key in dfs:
    dfs[key] = pd.concat(dfs[key], axis=0)

print("Sizes and uniqueness check:")
print({key: dfs[key].shape for key in dfs})
print({key: dfs[key]["reddit_post_id"].unique().shape[0] == dfs[key].shape[0] for key in dfs})

for key in dfs:
    dfs[key] = dfs[key].set_index("reddit_post_id")

df_merged = dfs["meta-llama"].join(
    [dfs["google"], dfs["Qwen"]],
    how="outer",
)

df_merged

In [ ]:
expected_keys = set({
  "Energy": "",
  "Materials": "",
  "Industrials": "",
  "Consumer Discretionary": "",
  "Consumer Staples": "",
  "Health Care": "",
  "Financials": "",
  "Information Technology": "",
  "Communication Services": "",
  "Utilities": "",
  "Real Estate": "",
  "rationale": "",
  "confidence": 0.0
}.keys())

In [ ]:
# real filter
new_df = {}
for i, row in df_merged.iterrows():
    a = row["google/gemma-7b-it"]
    b = row["Qwen/Qwen2.5-7B-Instruct"]
    c = row["meta-llama/Llama-3.1-8B-Instruct"]

    sectors = expected_keys - set(["rationale", "confidence",])
    valid = ["neutral", "negative", "positive"]

    save = True
    if pd.notna(a) and pd.notna(b) and pd.notna(c):
        for sector in sectors:
            if a.get(sector, None) in valid and b.get(sector, None) in valid and c.get(sector, None) in valid:
                pass
            else:
                save = False
                break
        
        if save and i not in new_df:
            new_df[i] = row

new_df = pd.DataFrame.from_dict(new_df, orient="index")
print("After filtering errors and Nans: ", new_df.shape)
new_df.sample(3)

In [ ]:
sentiment_df = {"reddit_post_id": [],
                "positive": [],
                "negative": [],}

for i, row in new_df.iterrows():
    a = row["google/gemma-7b-it"]
    b = row["Qwen/Qwen2.5-7B-Instruct"]
    c = row["meta-llama/Llama-3.1-8B-Instruct"]

    sectors = expected_keys - set(["rationale", "confidence",])

    pos_count = {}
    neg_count = {}

    for sector in sectors:
        votes = [a.get(sector, None), b.get(sector, None), c.get(sector, None)]
        pos_votes = votes.count("positive")
        neg_votes = votes.count("negative")

        if pos_votes > 0:
            pos_count[sector] = pos_votes
        if neg_votes > 0:
            neg_count[sector] = neg_votes


    sentiment_df["reddit_post_id"].append(i)
    sentiment_df["positive"].append(pos_count)
    sentiment_df["negative"].append(neg_count)

sentiment_df = pd.DataFrame.from_dict(sentiment_df)

print("All neutral posts: ", ((sentiment_df["positive"].map(len) == 0) & (sentiment_df["negative"].map(len) == 0)).sum())

sentiment_df_meaningful = sentiment_df[~((sentiment_df["positive"].map(len) == 0) & (sentiment_df["negative"].map(len) == 0))]
sentiment_df_meaningful.reset_index(drop=True, inplace=True)
sentiment_df_meaningful

In [ ]:
ranks = {sector: {"pos3": 0, "pos2": 0, "pos1": 0, " ": " ", "neu3": 0, "neu2": 0, "neu1": 0, "": "", "neg3": 0, "neg2": 0, "neg1": 0} for sector in sectors}

for i, row in new_df.iterrows():
    a = row["google/gemma-7b-it"]
    b = row["Qwen/Qwen2.5-7B-Instruct"]
    c = row["meta-llama/Llama-3.1-8B-Instruct"]

    sectors = expected_keys - set(["rationale", "confidence",])

    pos_count = {}
    neg_count = {}

    for sector in sectors:
        votes = [a.get(sector, None), b.get(sector, None), c.get(sector, None)]
        pos_votes = votes.count("positive")
        neg_votes = votes.count("negative")
        neu_votes = votes.count("neutral")

        if pos_votes == 3:
            ranks[sector]["pos3"] += 1
        elif pos_votes == 2:
            ranks[sector]["pos2"] += 1
        elif pos_votes == 1:
            ranks[sector]["pos1"] += 1

        if neu_votes == 3:
            ranks[sector]["neu3"] += 1
        elif neu_votes == 2:
            ranks[sector]["neu2"] += 1
        elif neu_votes == 1:
            ranks[sector]["neu1"] += 1

        if neg_votes == 3:
            ranks[sector]["neg3"] += 1
        elif neg_votes == 2:
            ranks[sector]["neg2"] += 1
        elif neg_votes == 1:
            ranks[sector]["neg1"] += 1
    
ranks_df = pd.DataFrame.from_dict(ranks, orient="index")
ranks_df

filtrar posts com as palavras ao setos nao serviu de nada

In [ ]:
ranks_df.sum(axis=0)

In [ ]:
ranks_df["pos"] = ranks_df["pos3"] + ranks_df["pos2"]
ranks_df["neu"] = ranks_df["neu3"] + ranks_df["neu2"]
ranks_df["neg"] = ranks_df["neg3"] + ranks_df["neg2"]

ranks_df[["pos", "neu", "neg"]]

------

------

------

criar o test set com base no gpt

In [ ]:
import pandas as pd
from collections import Counter

reddit_label = pd.read_parquet(str(PROJECT_ROOT / "02_sentiment/reddit/dataset.parquet"))

for_gpt_validation = {}

sectors = [x.split("_")[1] for x in reddit_label.columns if x.startswith("label_")]
for sector in sectors:
    for sentiment in ["positive", "neutral", "negative"]:
        
        total_3 = 0
        total_2 = 0

        random_order = reddit_label.sample(frac=1)

        for i, row in random_order.iterrows():
            votes = list(row[f"label_{sector}"])
            sentiment_votes = votes.count(sentiment)

            if sentiment_votes == 3 and total_3 < 20:
                label_dict = {}

                for sec in sectors:
                    sec_votes = row[f"label_{sec}"]
                    counts = Counter(sec_votes)
                    most_common = counts.most_common()
                    s = "neutral" if most_common[0][1] == 1 else most_common[0][0]

                    label_dict[sec] = {
                        "agreement": most_common[0][1],
                        "expected_sentiment": s,
                        "gpt_sentiment": "todo"
                    }


                for_gpt_validation[i] = {
                    "thread": row["thread"],
                    "label": label_dict
                }
                total_3 += 1
            
            elif sentiment_votes == 2 and total_2 < 20:
                label_dict = {}

                for sec in sectors:
                    sec_votes = row[f"label_{sec}"]
                    counts = Counter(sec_votes)
                    most_common = counts.most_common()
                    s = "neutral" if most_common[0][1] == 1 else most_common[0][0]

                    label_dict[sec] = {
                        "agreement": most_common[0][1],
                        "expected_sentiment": s,
                        "gpt_sentiment": "todo"
                    }

                for_gpt_validation[i] = {
                    "thread": row["thread"],
                    "label": label_dict
                }
                total_2 += 1
            
import json
# with open("reddit_gpt_validation.json", "w") as f:
#     json.dump(for_gpt_validation, f, indent=2)

In [ ]:
with open("reddit_gpt_validation.json", "r") as f:
    reddit_gpt_validation = json.load(f)

contagem_neg_3 = {}
contagem_pos_3 = {}
contagem_neg_2 = {}
contagem_pos_2 = {}
for post in reddit_gpt_validation.values():
    for sector in post["label"]:
        if post["label"][sector]["expected_sentiment"] == "negative":
            if post["label"][sector]["agreement"] == 3:
                contagem_neg_3[sector] = contagem_neg_3.get(sector, 0) + 1
            elif post["label"][sector]["agreement"] == 2:
                contagem_neg_2[sector] = contagem_neg_2.get(sector, 0) + 1
        elif post["label"][sector]["expected_sentiment"] == "positive":
            if post["label"][sector]["agreement"] == 3:
                contagem_pos_3[sector] = contagem_pos_3.get(sector, 0) + 1
            elif post["label"][sector]["agreement"] == 2:
                contagem_pos_2[sector] = contagem_pos_2.get(sector, 0) + 1

print("INCOMPLETOS due to under 20 samples per sector:")
print("Negative with 3 votes:", {k:v for k,v in contagem_neg_3.items() if v < 20})
print("Negative with 2 votes:", {k:v for k,v in contagem_neg_2.items() if v < 20})
print("Positive with 3 votes:", {k:v for k,v in contagem_pos_3.items() if v < 20})
print("Positive with 2 votes:", {k:v for k,v in contagem_pos_2.items() if v < 20})

In [ ]:
len(for_gpt_validation)

In [ ]:
# # API EM BAIXO ?!?!?!?
# # API EM BAIXO ?!?!?!?
# # API EM BAIXO ?!?!?!?
# # API EM BAIXO ?!?!?!?
# # API EM BAIXO ?!?!?!?

# import requests
# import json
# import re
# import time

# models_used = set()
# done_counter = 0

# with open(str(PROJECT_ROOT / "01_data/processed/reddit_gpt_validation.json"), "r") as f:
#     sector_gpt_api = json.load(f)

# import uuid



# for post_id in sector_gpt_api:
#     content = sector_gpt_api[post_id]

#     endpoint = "https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream"

#     headers = {
#         "x-api-key": IAEDU_API_KEY  # from src.secrets (see src/secrets_example.py)
#     }

#     def prompt_template(content):
#         thread = content["thread"]
#         sectors = list(content["label"].keys())

#         x = (
#             "Classify the sentiment of the following Reddit post towards the specified sectors. Return a JSON object with the sentiment for each sector.\n"
#             f"Post: {thread}\n"
#             f"Sectors: {', '.join(sectors)}\n"
#         )
#         return x

#     thread_id = str(uuid.uuid4())
#     data = {
#         "channel_id": "cmkpzn9wrjui7gr01gz7qvdsu",
#         "thread_id": thread_id,
#         "user_info": "{}",
#         "message": prompt_template(content=content)
#     }

#     response = requests.post(
#         endpoint,
#         headers=headers,
#         data=data,
#         stream=True
#     )

#     final_answer = None

#     for line in response.iter_lines(decode_unicode=True):
#         if not line:
#             continue

#         event = json.loads(line)

#         print(event)

#         if event["type"] == "message":
#             final_answer = event["content"]["content"]
#             model_name = event["content"]["response_metadata"]["model_name"]

#         elif event["type"] == "done":
#             break

#         # print(event)
#     try:
#         print("Final answer:", final_answer)

#         sentiment = "positive"

#         #sentiment = match.group(1).lower() if match else "todo"
    
        
#     except Exception as e:
#         sentiment = "todo"

#     if sentiment == "todo":
#         print(f"Could not extract sentiment for item: {item} at {time.strftime('%H:%M:%S', time.gmtime())}.")
#         continue

#     else:
#         #sector_gpt_api[post_id]["label"] = sentiment
#         pass

#     done_counter += 1
#     print(f"Done items: {done_counter}", end="\r")
#     models_used.add(model_name)



# # with open("", "w") as f:
# #     json.dump(sector_gpt_api, f, indent=2)
    
# print("Models used:", models_used)

# # API EM BAIXO DO GPT!

In [ ]:
# import json
# import pyperclip
# import random

# current_version = 43

# with open("reddit_gpt_validation.json", "r") as f:
#     reddit_gpt_validation = json.load(f)

# with open(f"reddit_gpt_validation_v{current_version}.json", "r") as f:
#     reddit_gpt_validation_with_labels = json.load(f)

# sectors = {'Communication Services',
#  'Consumer Discretionary',
#  'Consumer Staples',
#  'Energy',
#  'Financials',
#  'Health Care',
#  'Industrials',
#  'Information Technology',
#  'Materials',
#  'Real Estate',
#  'Utilities'}

# # copy the following mesage to clipboard
# message = (
#     "You are a financial-domain impact classifier for equity sectors. Your task is to analyze a Reddit post and classify the expected impact of the described events on specific GICS sectors\n."
#     "Classification Rules:\n"
#     '- "positive": applies only if the thread expresses MEANINGFUL bullishness, optimism, expected gains, or favorable news for a sector.\n'
#     '- "negative": applies only if the thread expresses MEANINGFUL bearishness, pessimism, expected declines, or harmful news for a sector.\n'
#     '- "neutral": applies when the thread does not meaningfully reference the sector, sentiment is unclear, or discussion is casual/speculative without substance.\n'
#     "\n"
#     'Definition of "meaningful":\n'
#     "- A clear expectation of price movement or performance (up or down)\n"
#     "- A stated reason, catalyst, or justification (e.g., earnings, news, macro factors, valuation)\n"
#     "- Repeated or emphasized sentiment across multiple comments\n"
#     "- Explicit long or short positioning with intent\n"
#     "\n"
#     'The following are NOT meaningful by themselves:\n'
#     "- Casual mentions of sectors without sentiment (e.g., answering what lists of stocks, and stuff like that)\n"
#     "- Jokes, memes, sarcasm, or one-line opinions\n"
#     "- Watchlists, tickers without commentary, or gap lists\n"
#     '- Statements without reasoning (e.g., "this stock will moon" or "short it" without explanation)\n'
#     "\n"
#     "Output Requirements: Return a single JSON object with the sentiment for each sector mentioned in the post. The JSON should have the format:\n"
#     "{\n"
#     '   "Communication Services": "<positive | negative | neutral>",\n'
#     '   "Consumer Discretionary": "<positive | negative | neutral>",\n'
#     '   "Consumer Staples": "<positive | negative | neutral>",\n'
#     '   "Energy": "<positive | negative | neutral>",\n'
#     '   "Financials": "<positive | negative | neutral>",\n'
#     '   "Health Care": "<positive | negative | neutral>",\n'
#     '   "Industrials": "<positive | negative | neutral>",\n'
#     '   "Information Technology": "<positive | negative | neutral>",\n'
#     '   "Materials": "<positive | negative | neutral>",\n'
#     '   "Real Estate": "<positive | negative | neutral>",\n'
#     '   "Utilities": "<positive | negative | neutral>"\n'
#     "}\n"
#     "Classify strictly according to these rules. Output only the JSON object, nothing else. Are you ready?"
# )

# pyperclip.copy(message)

# ready = input("Message copied to clipboard. Are you ready to classify the Reddit posts?")
# post_ids = list(reddit_gpt_validation.keys())
# random.shuffle(post_ids)

# total_posts = len(post_ids)
# current_done = len(reddit_gpt_validation_with_labels)
# done_in_session = 0
# missing_posts = total_posts - current_done

# for post_id in post_ids:

#     if post_id in reddit_gpt_validation_with_labels:
#         continue

#     content = reddit_gpt_validation[post_id]
    
#     thread = content["thread"]
#     pyperclip.copy(thread)

#     lines = thread.split('\n')
#     gpt_answer = input(f"Title: {lines[2]}")

#     if gpt_answer.lower() in ("end", "quit", "exit"):
#         break

#     try:
#         classification = json.loads(gpt_answer)
#     except json.JSONDecodeError:
#         print("Invalid JSON. Please try again.")
#         continue

#     # Validate that all sectors are present in the classification
#     missing_sectors = sectors - set(classification.keys())
#     if missing_sectors:
#         print(f"Missing sectors: {missing_sectors}")
#         continue

#     # validate that all values are valid sentiments
#     valid_sentiments = {"positive", "negative", "neutral"}
#     invalid_sectors = [sector for sector, sentiment in classification.items() if sentiment not in valid_sentiments]
#     if invalid_sectors:
#         print(f"Invalid sentiments for sectors: {invalid_sectors}")
#         continue

#     reddit_gpt_validation_with_labels[post_id] = reddit_gpt_validation[post_id]

#     for sector in reddit_gpt_validation_with_labels[post_id]["label"]:
#         reddit_gpt_validation_with_labels[post_id]["label"][sector]["gpt_sentiment"] = classification[sector]

#     done_in_session += 1
#     print(f"Done {done_in_session}/{missing_posts} posts ({done_in_session/missing_posts*100:.1f}%)")


# new_version = current_version + 1

# input(f"Sure you want to save the results as v{new_version}?")
# input(f"Sure you want to save the results as v{new_version}?")

# with open(f"reddit_gpt_validation_v{new_version}.json", "w") as f:
#     json.dump(reddit_gpt_validation_with_labels, f, indent=2)

gpt 5.2 again

In [ ]:
import json
import pandas as pd

#with open(f"reddit_gpt_validation_v47.json", "r") as f:
with open(f"reddit_gpt_validation_final.json", "r") as f:
    reddit_gpt_validation_with_labels = json.load(f)

concordancia_df = {}

for post_id, content in reddit_gpt_validation_with_labels.items():
    for sector in content["label"]:
        expected = content["label"][sector]["expected_sentiment"]
        gpt_label = content["label"][sector]["gpt_sentiment"]
        agreement = content["label"][sector]["agreement"]

        if sector not in concordancia_df:
            # number of votes (2 or 3) + expected sentiment
            concordancia_df[sector] = {"2 (negative)": {"match": 0, "total": 0},
                                       "3 (negative)": {"match": 0, "total": 0},
                                       "1 (neutral)": {"match": 0, "total": 0},
                                       "2 (neutral)": {"match": 0, "total": 0},
                                       "3 (neutral)": {"match": 0, "total": 0},
                                       "2 (positive)": {"match": 0, "total": 0},
                                       "3 (positive)": {"match": 0, "total": 0},}
        
        concordancia_df[sector][f"{agreement} ({expected})"]["total"] += 1
        if expected == gpt_label:
            concordancia_df[sector][f"{agreement} ({expected})"]["match"] += 1

for sector in concordancia_df:
    for agreement in concordancia_df[sector]:
        match = concordancia_df[sector][agreement]["match"]
        total = concordancia_df[sector][agreement]["total"]
        pct = (match / total * 100) if total > 0 else 0
        concordancia_df[sector][agreement] = f"{pct:.1f}% ({match}/{total})"

concordancia_df = pd.DataFrame.from_dict(concordancia_df, orient="index")

#meaan per column
print("Mean per column:")
for column in concordancia_df.columns:
    matches = concordancia_df[column].apply(lambda x: int(x.split("(")[1].split("/")[0]))
    total = concordancia_df[column].apply(lambda x: int(x.split("(")[1].split("/")[1].split(")")[0]))
    mean_pct = (matches.sum() / total.sum() * 100) if total.sum() > 0 else 0
    print(f"{column}: {mean_pct:.1f}%")

concordancia_df

In [ ]:
concordancia_df.columns

In [ ]:
import pandas as pd
import re

# function to extract numerator and denominator
def extract_counts(cell):
    if pd.isna(cell):
        return 0, 0
    match = re.search(r"\((\d+)/(\d+)\)", str(cell))
    if match:
        return int(match.group(1)), int(match.group(2))
    return 0, 0

# define your columns
cols_2 = ["2 (negative)", "2 (neutral)", "2 (positive)"]
cols_3 = ["3 (negative)", "3 (neutral)", "3 (positive)"]

def compute_weighted_mean(df, cols):
    total_success = 0
    total_n = 0
    
    for col in cols:
        counts = df[col].apply(extract_counts)
        total_success += sum(x[0] for x in counts)
        total_n += sum(x[1] for x in counts)
    
    return total_success / total_n if total_n > 0 else None

mean_2 = compute_weighted_mean(concordancia_df, cols_2)
mean_3 = compute_weighted_mean(concordancia_df, cols_3)

print(f"Mean for vote=2: {mean_2:.4f} ({mean_2*100:.2f}%)")
print(f"Mean for vote=3: {mean_3:.4f} ({mean_3*100:.2f}%)")